# L1 — отдельные ряды на временной сетке потоков

Для каждого источника L0 выпускается отдельный CSV за весь период. Мачтовые ряды
именуются `<площадка>-<высота>_fluxes.csv`, `<площадка>-<высота>_meteo.csv`
и `<площадка>-<высота>_radiation.csv`. Для отдельных приборов АМИИС используются
`TFN-IMCES-meteo.csv` и `TFN-IMCES-radiation.csv`. Файлы лежат в папках городов
`processed/L1/Moscow`, `processed/L1/Tomsk` и `processed/L1/Beijing`.

Шаг и положение сетки определяем по временным меткам потоков. Сначала ищем
потоки того же уровня, затем потоки площадки или города с общей сеткой.
В каждом файле охватываем период соответствующего источника. Метеозначения
берём на точных временных метках без интерполяции. Радиацию усредняем по
наличным значениям за окно `(t − шаг, t]`. Дополнительные метеоуровни
сохраняются также там, где на этой высоте нет потоков.

Метки потоков и значения сохраняются. Для Звенигорода используется `sonic-fluxes`,
а записи вне получасовых границ исключаются по ранее согласованному правилу.
Исключённые метки сохраняются в диагностике. ЛОД и вспомогательные поля АМИИС
остаются вне состава L1.

Для потоковых CSV рассчитываем `Cd = (Ustar/Wind)²` и флаги контроля качества.
Проверяем физические диапазоны `Qh`, `Qle`, `Qtau` и `Cd`, затем оцениваем
выбросы по часовым статистикам в 60-дневном окне. Пороги составляют 5σ для
потоков и 10σ для `Cd`. В CSV записываем `qc_tot`, `qc_tot_noCd` и их варианты
`_ext` с часовым расширением. Флаг 0 отмечает запись, отклонённую проверкой.
Пропуск сам по себе даёт флаг 1, как в прежнем алгоритме. Исходные значения
потоков сохраняются, в том числе при флаге 0. Параметры проверки заданы
в `flux_qc.py`.


In [ ]:
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import pandas as pd
from IPython.display import display

from l0_data import PROCESSED_ROOT
from l1_data import source_configs
from l1_publish import stage_l1, publish_l1

processed_root = PROCESSED_ROOT
work_root = Path('_temp').resolve()
stamp = datetime.now(ZoneInfo('Europe/Moscow')).strftime('%Y%m%d_%H%M%S_%f')
run_dir = work_root / f'l1_sources_{stamp}'
VERIFY_RADIATION = True

sources = source_configs(processed_root)
display(pd.DataFrame([
    {'город': s.city, 'площадка': s.site, 'тип': s.kind,
     'источник L0': s.path.relative_to(processed_root).as_posix(),
     'CSV L1': s.output_path(processed_root).relative_to(processed_root).as_posix()}
    for s in sources
]))


## Подготовка набора

Копия текущих файлов L1 и отчётов сохраняется в папке рабочего запуска.
Новые ряды сначала вычисляются там же. Для метеоданных сверяются точные
метки и значения, для радиации независимо проверяются число отсчётов и средние
во всех окнах, включая переходы между месяцами. Подробности остаются
в диагностике `reports/L1`.

Число строк, отмеченных каждым итоговым флагом QC, приводится в сводке L1.


In [ ]:
summary = stage_l1(run_dir, processed_root, verify_radiation=VERIFY_RADIATION)
display(summary.groupby(['city', 'kind', 'interval_minutes']).size().rename('files').reset_index())
display(summary)


## Сохранение L1

После проверок новый набор заменяет прежние файлы L1 и их отчёты. Резервная
копия прежнего набора остаётся в `_temp`.


In [ ]:
published = publish_l1(run_dir, processed_root)
print('CSV уровня L1:', published['files'])
print('Папка данных:', processed_root / 'L1')
print('Резервная копия:', published['backup'])
